In [4]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
import seaborn as sns
import nltk
import re
import tensorflow as tf

import nltk
nltk.download('stopwords')
nltk.download('punkt_tab')
nltk.download('wordnet')
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
from sklearn.feature_extraction.text import TfidfVectorizer
from nltk.tokenize import word_tokenize


from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder,MultiLabelBinarizer
from sklearn.metrics import classification_report,hamming_loss


from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding,LSTM,Dense,Dropout,Bidirectional,GRU,SimpleRNN
from tensorflow.keras.callbacks import EarlyStopping
from tensorflow.keras.utils import to_categorical

[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\haris\AppData\Roaming\nltk_data...
[nltk_data]   Unzipping corpora\stopwords.zip.
[nltk_data] Downloading package punkt_tab to
[nltk_data]     C:\Users\haris\AppData\Roaming\nltk_data...
[nltk_data]   Unzipping tokenizers\punkt_tab.zip.
[nltk_data] Downloading package wordnet to
[nltk_data]     C:\Users\haris\AppData\Roaming\nltk_data...


In [6]:
df=pd.read_csv(r"C:\Users\haris\.vscode\comment_toxicity_DL\train.csv")

In [7]:
df.head()

,id,comment_text,toxic,severe_toxic,obscene,threat,insult,identity_hate
0,0000997932d777bf,Explanation\nWhy the edits made under my usern...,0,0,0,0,0,0
1,000103f0d9cfb60f,D'aww! He matches this background colour I'm s...,0,0,0,0,0,0
2,000113f07ec002fd,"Hey man, I'm really not trying to edit war. It...",0,0,0,0,0,0
3,0001b41b1c6bb37e,"""\nMore\nI can't make any real suggestions on ...",0,0,0,0,0,0
4,0001d958c54c6e35,"You, sir, are my hero. Any chance you remember...",0,0,0,0,0,0


In [10]:
for col in df:
    print(df[col].value_counts())

id
0000997932d777bf    1
000103f0d9cfb60f    1
000113f07ec002fd    1
0001b41b1c6bb37e    1
0001d958c54c6e35    1
                   ..
ffe987279560d7ff    1
ffea4adeee384e90    1
ffee36eab5c267c9    1
fff125370e4aaaf3    1
fff46fc426af1f9a    1
Name: count, Length: 159571, dtype: int64
comment_text
Explanation\nWhy the edits made under my username Hardcore Metallica Fan were reverted? They weren't vandalisms, just closure on some GAs after I voted at New York Dolls FAC. And please don't remove the template from the talk page since I'm retired now.89.205.38.27                                                                                                                                                                                                                                                                                                                                                                             1
D'aww! He matches this background colour I'm seemingly stuck with. T

In [12]:
df.drop(columns='id',inplace=True)

In [14]:
df.columns

Index(['comment_text', 'toxic', 'severe_toxic', 'obscene', 'threat', 'insult',
       'identity_hate'],
      dtype='object')

In [15]:
lemmetizer=WordNetLemmatizer()
stop_words=set(stopwords.words('english'))


def clean_text(text):
  text=re.sub(r'[^a-zA-Z]',' ',text)
  text=text.lower()
  text=word_tokenize(text)
  text=[lemmetizer.lemmatize(word) for word in text if word not in stop_words]
  text=' '.join(text)
  return text


df["clean_text"]=df["comment_text"].apply(clean_text)

In [16]:
Y_label=df[['toxic', 'severe_toxic', 'obscene', 'threat', 'insult',
       'identity_hate']].values

In [17]:
Y_label

array([[0, 0, 0, 0, 0, 0],
       [0, 0, 0, 0, 0, 0],
       [0, 0, 0, 0, 0, 0],
       ...,
       [0, 0, 0, 0, 0, 0],
       [0, 0, 0, 0, 0, 0],
       [0, 0, 0, 0, 0, 0]], shape=(159571, 6))

In [18]:
X_train,X_test,y_train,y_test=train_test_split(df['clean_text'],Y_label,test_size=0.2,random_state=42)

In [25]:
Vocb_Size=20000
Max_len=100

tokenizer=Tokenizer(num_words=Vocb_Size,oov_token='<OOV>')
tokenizer.fit_on_texts(X_train)


import pickle
pickle.dump(tokenizer, open("tokenizer.pkl", "wb"))

X_train_seq = tokenizer.texts_to_sequences(X_train)
X_test_seq = tokenizer.texts_to_sequences(X_test)

X_train_pad = pad_sequences(X_train_seq, maxlen=Max_len, padding='post')
X_test_pad = pad_sequences(X_test_seq, maxlen=Max_len, padding='post')

X_train_pad

array([[10639,  6992,  3034, ...,     0,     0,     0],
       [   21,   101,     8, ...,     0,     0,     0],
       [ 6371,   981,  4940, ...,   383,  5398,     1],
       ...,
       [   94,   801,  4094, ...,     0,     0,     0],
       [  505,   401,    12, ...,     0,     0,     0],
       [  147,  3938,  5728, ...,     0,     0,     0]],
      shape=(127656, 100), dtype=int32)

In [20]:
label_model=Sequential(
    [Embedding(Vocb_Size,64,input_length=Max_len),
    Bidirectional(LSTM(128)),
    Dropout(0.3),
    Dense(64,activation='relu'),
    Dropout(0.2),
    Dense(6,activation='sigmoid')]
)

label_model.compile(
    optimizer='adam',
    loss='binary_crossentropy',
    metrics=[tf.keras.metrics.AUC(name='auc')]
)
label_model.summary()

c:\Users\haris\AppData\Local\Programs\Python\Python312\Lib\site-packages\keras\src\layers\core\embedding.py:103: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding (Embedding)           │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ bidirectional (Bidirectional)   │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ ?                      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ ?                      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ ?                      │   0 (unbuilt) │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 0 (0.00 B)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 0 (0.00 B)

In [21]:
early_stop = EarlyStopping(
    monitor              = 'val_loss',
    patience             = 3,
    restore_best_weights = True
)

history_topic = label_model.fit(
    X_train_pad, y_train,
    validation_data=(X_test_pad, y_test),
    epochs=10,
    batch_size=64,
    callbacks=[early_stop]
)

Epoch 1/10
1995/1995 ━━━━━━━━━━━━━━━━━━━━ 330s 163ms/step - auc: 0.9583 - loss: 0.0691 - val_auc: 0.9805 - val_loss: 0.0498
Epoch 2/10
1995/1995 ━━━━━━━━━━━━━━━━━━━━ 367s 184ms/step - auc: 0.9821 - loss: 0.0473 - val_auc: 0.9800 - val_loss: 0.0482
Epoch 3/10
1995/1995 ━━━━━━━━━━━━━━━━━━━━ 350s 175ms/step - auc: 0.9863 - loss: 0.0423 - val_auc: 0.9791 - val_loss: 0.0484
Epoch 4/10
1995/1995 ━━━━━━━━━━━━━━━━━━━━ 339s 170ms/step - auc: 0.9897 - loss: 0.0377 - val_auc: 0.9724 - val_loss: 0.0517
Epoch 5/10
1995/1995 ━━━━━━━━━━━━━━━━━━━━ 348s 175ms/step - auc: 0.9919 - loss: 0.0339 - val_auc: 0.9670 - val_loss: 0.0555


In [22]:
Y_train_pred= (label_model.predict(X_train_pad,verbose=0)>=0.5).astype(int)
Y_test_pred= (label_model.predict(X_test_pad,verbose=0)>=0.5).astype(int)

In [23]:
Train_Hamming_loss=hamming_loss(y_train,Y_train_pred)
Test_Hamming_loss=hamming_loss(y_test,Y_test_pred)

print('Train Hamming Loss',Train_Hamming_loss*100)
print('Test Hamming Loss',Test_Hamming_loss*100)


Train Hamming Loss 1.5781997451484195
Test Hamming Loss 1.8016606611311294


In [24]:
label_model.save("lstm_model.h5")

In [ ]:
labels = ["toxic", "severe_toxic", "obscene", "threat", "insult", "identity_hate"]

def predict_labels(sentence):
    text = clean_text(sentence)
    seq = tokenizer.texts_to_sequences([text])
    padded = pad_sequences(seq, maxlen=Max_len, padding='post')

    pred = label_model.predict(padded, verbose=0)[0]  # remove batch dimension

    result = (pred > 0.2).astype(int)

    return [labels[i] for i in range(len(labels)) if result[i] == 1]

In [ ]:
predict_labels("i will kill you")

In [ ]:
def predict_labels_batch(texts, batch_size=512):
    
    # Step 1 - Clean all texts at once
    cleaned = [clean_text(t) for t in texts]
    
    # Step 2 - Tokenize all at once
    seqs = tokenizer.texts_to_sequences(cleaned)
    
    # Step 3 - Pad all at once
    padded = pad_sequences(seqs, maxlen=Max_len, padding='post')
    
    # Step 4 - Predict in batches
    preds = label_model.predict(padded, batch_size=batch_size, verbose=1)
    
    # Step 5 - Apply threshold and map labels
    results = (preds > 0.2).astype(int)
    return [
        ", ".join([label for label, res in zip(labels, row) if res == 1]) or "clean"
        for row in results
    ]

# Apply
df_test=pd.read_csv("/test_data.csv")
df_test["test_data"] = predict_labels_batch(df_test["comment_text"])

In [ ]:
'''' for small data-df_test["test_data"] = df_test["comment_text"].apply(predict_labels)'''

In [ ]:
df_test.head()

In [ ]:
'''SIMPLE RNN MODEL'''

from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, SimpleRNN, Dense

model_rnn = Sequential([
    Embedding(input_dim=20000, output_dim=64, input_length=100),
    SimpleRNN(64),
    Dense(6, activation='sigmoid')
])

model_rnn.compile(
    optimizer='adam',
    loss='binary_crossentropy',
    metrics=[tf.keras.metrics.AUC(name='auc')]
)

In [ ]:
early_stop = EarlyStopping(
    monitor              = 'val_loss',
    patience             = 3,
    restore_best_weights = True
)

history_topic = model_rnn.fit(
    X_train_pad, y_train,
    validation_data=(X_test_pad, y_test),
    epochs=10,
    batch_size=64,
    callbacks=[early_stop]
)

In [ ]:
'''' BERT MODEL'''

In [ ]:
! pip install transformers==4.41.1

In [ ]:
X = df["comment_text"]
y = df[["toxic","severe_toxic","obscene","threat","insult","identity_hate"]].values

In [ ]:
print(X.dtype)
print(y.dtype)

In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42
)

In [ ]:
from transformers import BertTokenizer

tokenizer = BertTokenizer.from_pretrained("bert-base-uncased")

In [ ]:
train_encodings = tokenizer(
    list(X_train),
    truncation=True,
    padding=True,
    max_length=128,
    return_tensors='tf'   # tensorflow tensors
)

# Step 2 — Encode Test
test_encodings = tokenizer(
    list(X_test),
    truncation=True,
    padding=True,
    max_length=128,
    return_tensors='tf'
)

In [ ]:
import numpy as np

y_train = np.array(y_train).astype("float32")
y_test = np.array(y_test).astype("float32")

In [ ]:
import tensorflow as tf

train_dataset = tf.data.Dataset.from_tensor_slices((
    dict(train_encodings),
    y_train
    )).batch(16)


test_dataset = tf.data.Dataset.from_tensor_slices((
    dict(test_encodings),
    y_test
)).batch(16)

In [ ]:
import tensorflow as tf
from transformers import TFBertModel
from tensorflow.keras.layers import Dense, Dropout
from tensorflow.keras.models import Model

bert = TFBertModel.from_pretrained("bert-base-uncased")

class BertLayer(tf.keras.layers.Layer):
    def __init__(self, bert_model):
        super().__init__()
        self.bert = bert_model

    def call(self, inputs):
        input_ids, attention_mask = inputs
        output = self.bert(input_ids=input_ids,
                           attention_mask=attention_mask)
        return output[0] 

In [ ]:
input_ids = tf.keras.Input(shape=(128,), dtype=tf.int32, name="input_ids")
attention_mask = tf.keras.Input(shape=(128,), dtype=tf.int32, name="attention_mask")

bert_layer = BertLayer(bert)

sequence_output = bert_layer([input_ids, attention_mask])
cls_token = sequence_output[:, 0, :]

x = Dense(128, activation="relu")(cls_token)
x = Dropout(0.3)(x)
x = Dense(64, activation="relu")(x)

output = Dense(6, activation="sigmoid")(x)

model = Model(inputs=[input_ids, attention_mask], outputs=output)

In [ ]:
model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=2e-5),
    loss="binary_crossentropy",
    metrics=[tf.keras.metrics.AUC(name="auc"),"accuracy"]
)

In [ ]:
history = model.fit(
    train_dataset,
    validation_data=test_dataset,
    epochs=3,
    callbacks=[early_stop]
)

In [ ]:
def predict_labels(text):
    enc = tokenizer(
        text,
        truncation=True,
        padding="max_length",
        max_length=128,
        return_tensors="tf"
    )

    preds = model.predict({
        "input_ids": enc["input_ids"],
        "attention_mask": enc["attention_mask"]
    }, verbose=0)[0]

    labels = ["toxic","severe_toxic","obscene","threat","insult","identity_hate"]

    return [labels[i] for i in range(6) if preds[i] > 0.3]

In [ ]:
predict_labels("i HATE YOU ")